# Step 20: Pretrained Transformers (frozen) + trained head
The pretrained Transformer is **not changed**; only a small model on top of its features is trained: XGBoost or a dense network.
*Same as `src/20_pretrained_transformers.py`. Slow on a CPU (about 45 minutes).*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page: https://multimediate-challenge.org/datasets/Dataset_NoXi/
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017, https://doi.org/10.1145/3136755.3136780
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

SETS = {   # name -> feature files
    "XLM-RoBERTa": ["xlm_roberta"],
    "w2v-BERT 2.0": ["w2vbert2"],
    "CLIP": ["clip"],
    "DINOv2": ["dino"],
    "Swin": ["swin"],
    "VideoMAE": ["videomae"],
    "w2v-BERT 2.0 + VideoMAE": ["w2vbert2", "videomae"],
}

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Head 1: XGBoost.**  **Head 2: dense network** (256-128, dropout 0.1). Both: scale on training rows, train, return the CCC on the test rows.

In [ ]:
def xgb_score(train, test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train])
    X_test = scaler.transform(X[test])
    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                         colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1)
    model.fit(X_train, y[train])
    return ccc(y[test], model.predict(X_test))


def dense_score(train, test):
    keras.utils.set_random_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(X.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    scaler = StandardScaler().fit(X[train])
    model.fit(np.clip(scaler.transform(X[train]), -5, 5), y[train], epochs=30, batch_size=128, verbose=0)
    return ccc(y[test], model.predict(np.clip(scaler.transform(X[test]), -5, 5), verbose=0).flatten())

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement", "language", "session"])
y = df["engagement"].to_numpy("float32")
language = df["language"].to_numpy()
session = df["session"].to_numpy()
languages = ["English", "French", "German"]

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**For each pretrained model:** 3 folds with both heads, then within-language and cross-language (XGBoost).

In [ ]:
for name, files in SETS.items():
    parts = []
    for f in files:
        parts.append(np.load("data/features/pretrained_" + f + ".npy"))
    X = np.concatenate(parts, axis=1)

    xgb_scores, dense_scores = [], []
    for test_sessions in folds:
        test = np.isin(session, test_sessions)
        xgb_scores.append(xgb_score(~test, test))
        dense_scores.append(dense_score(~test, test))

    within = []
    for l in languages:
        for s in sorted(df[df["language"] == l]["session"].unique()):
            within.append(xgb_score((language == l) & (session != s), session == s))
    cross = []
    for a in languages:
        for b in languages:
            if a != b:
                cross.append(xgb_score(language == a, language == b))

    drop = 100 * (np.mean(within) - np.mean(cross)) / np.mean(within)
    print(name.ljust(24), "XGB", round(np.mean(xgb_scores), 3), "| dense", round(np.mean(dense_scores), 3),
          "| drop", round(drop), "%")

**The result saved by the script**, for comparison.

In [ ]:
print(pd.read_csv("reports/pretrained_results.csv")[["model", "xgb_ccc", "dense_ccc", "drop_pct"]].round(3))